# Revision analyses for the JBHI submission

This notebook adds four light analyses requested in review of the Paper 1 manuscript. It does not
change any feature, transformation, model route, or multiplicity rule from Notebooks 02 and 03.

| Section | Question | Reviewer point |
|---|---|---|
| R1 | Do ALS–control differences persist when ALS recordings are restricted to normal bulbar function (ALSFRS-R bulbar 11–12)? | Phenotype vs acquisition |
| R2 | Does measure availability differ by cohort and bulbar severity group? | Support-dependent missingness |
| R3 | How are cohort and bulbar effects, and persistence (ICC), distributed across separability classes A/B/C, relative to the base rate? | Separability interpretation |
| R4 | Is the within- vs between-family contrast robust to excluding all shared-support pairs and to leaving out each family? | Dimensional structure |

**Prerequisites.** Run Notebooks 02 and 03 to completion with `FINAL_RUN = True` first. This notebook
reads their output tables and re-executes Notebook 02's input, harmonization, scaling, and model-helper
cells (sections 0–6 and 8) in an isolated namespace, writing that rebuild under
`outputs/revision_analyses/nb02_rebuild/` so Notebook 02's own outputs are never overwritten.

**Outputs.** `outputs/revision_analyses/tables/*.csv` and `outputs/revision_analyses/revision_summary.md`.
All outputs are aggregate (no participant-level rows) and can be shared for manuscript writing.

## 0. Configuration and prerequisite check

In [ ]:
from __future__ import annotations

import json
import os
import platform
import warnings
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.multitest import multipletests
from IPython.display import display

FINAL_RUN = True
SEED = 20260805
N_FAMILY_PERMUTATIONS = 10_000 if FINAL_RUN else 500
NORMAL_BULBAR_MIN = 11          # ALSFRS-R bulbar 11-12 = normal bulbar function
Q_THRESHOLD = 0.05


def locate_analysis_root() -> Path:
    configured = os.environ.get('PAPER1_ANALYSIS_ROOT', '').strip()
    if configured:
        return Path(configured).expanduser().resolve()
    start = Path.cwd().resolve()
    for candidate in [start, *start.parents]:
        if (candidate / 'requirements.txt').is_file() and (candidate / 'notebooks').is_dir():
            return candidate
    raise FileNotFoundError('Could not locate the analysis repository root')


ANALYSIS_ROOT = locate_analysis_root()
NB02_PATH = ANALYSIS_ROOT / 'notebooks' / '02_extent_distribution_persistence.ipynb'
NB02_TABLES = ANALYSIS_ROOT / 'outputs' / 'extent_distribution_persistence' / 'tables'
NB03_TABLES = ANALYSIS_ROOT / 'outputs' / 'multidimensional_structure' / 'tables'
OUTPUT_ROOT = ANALYSIS_ROOT / 'outputs' / 'revision_analyses'
TABLE_DIR = OUTPUT_ROOT / 'tables'
LOG_DIR = OUTPUT_ROOT / 'logs'
for directory in [TABLE_DIR, LOG_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

REQUIRED_INPUTS = {
    'nb02 cohort models': NB02_TABLES / 'cohort_models.csv',
    'nb02 bulbar models': NB02_TABLES / 'bulbar_models.csv',
    'nb02 persistence': NB02_TABLES / 'participant_level_persistence.csv',
    'nb02 availability models': NB02_TABLES / 'availability_models.csv',
    'nb03 pairwise table': NB03_TABLES / 'pairwise_relationship_classification.csv',
    'nb03 family contrast': NB03_TABLES / 'family_contrast_summary.csv',
}
missing = {name: str(path) for name, path in REQUIRED_INPUTS.items() if not path.is_file()}
if missing:
    raise FileNotFoundError(
        'Run Notebooks 02 and 03 (FINAL_RUN = True) before this notebook. Missing:\n'
        + '\n'.join(f'  {name}: {path}' for name, path in missing.items())
    )


def export(frame: pd.DataFrame, name: str) -> Path:
    path = TABLE_DIR / name
    frame.to_csv(path, index=False)
    return path


def bh(p_values: pd.Series) -> pd.Series:
    out = pd.Series(np.nan, index=p_values.index, dtype=float)
    mask = p_values.notna() & np.isfinite(p_values)
    if mask.any():
        out.loc[mask] = multipletests(p_values.loc[mask].astype(float), method='fdr_bh')[1]
    return out


print('ANALYSIS_ROOT:', ANALYSIS_ROOT)
print('OUTPUT_ROOT:  ', OUTPUT_ROOT)

## 1. Rebuild the Notebook 02 analysis matrix and model helpers

The cells are taken verbatim from Notebook 02 (located by their section headings), with two
substitutions: the output root is redirected, and `display` is silenced. The rebuilt matrix must pass
Notebook 02's own blocking population audit (519 recordings, 224 participants).

In [ ]:
NB02_SECTIONS = [
    '## 0. Frozen statistical specification',
    '## 1. Shared utilities',
    '## 2. Discover and hash immutable inputs',
    '## 3. Frozen feature registry',
    '## 4. Harmonize family tables',
    '## 5. Blocking population',
    '## 6. Availability, transforms',
    '## 8. Clustered-model helpers',
]
ORIGINAL_OUTPUT_LINE = "OUTPUT_ROOT = ANALYSIS_ROOT / 'outputs' / 'extent_distribution_persistence'"
REBUILD_OUTPUT_LINE = "OUTPUT_ROOT = ANALYSIS_ROOT / 'outputs' / 'revision_analyses' / 'nb02_rebuild'"
DISPLAY_IMPORT = 'from IPython.display import display, Markdown'

nb02 = json.loads(NB02_PATH.read_text(encoding='utf-8'))
nb02_cells = nb02['cells']


def code_after_heading(prefix: str) -> str:
    for index, cell in enumerate(nb02_cells):
        if cell['cell_type'] == 'markdown' and ''.join(cell['source']).lstrip().startswith(prefix):
            for following in nb02_cells[index + 1:]:
                if following['cell_type'] == 'code':
                    return ''.join(following['source'])
    raise KeyError(f'Notebook 02 section not found: {prefix!r}')


nb02_sources = [code_after_heading(prefix) for prefix in NB02_SECTIONS]
if ORIGINAL_OUTPUT_LINE not in nb02_sources[0] or DISPLAY_IMPORT not in nb02_sources[0]:
    raise AssertionError('Notebook 02 setup cell changed; update the rebuild substitutions.')
nb02_sources[0] = (
    nb02_sources[0]
    .replace(ORIGINAL_OUTPUT_LINE, REBUILD_OUTPUT_LINE)
    .replace(DISPLAY_IMPORT, '')
)

nb02_ns: dict = {'__name__': 'nb02_rebuild', 'display': lambda *args, **kwargs: None,
                 'Markdown': str}
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    for prefix, source in zip(NB02_SECTIONS, nb02_sources):
        exec(compile(source, f'<nb02 {prefix}>', 'exec'), nb02_ns)

analysis = nb02_ns['analysis']
FEATURE_REGISTRY = nb02_ns['FEATURE_REGISTRY']
fit_continuous_clustered = nb02_ns['fit_continuous_clustered']
fit_binary_gee = nb02_ns['fit_binary_gee']

assert len(analysis) == 519 and analysis['participant_id'].nunique() == 224
print('Rebuilt analysis matrix:', analysis.shape)

## 2. Separability classes and the 14 main outputs

Classes follow from Eq. 1 and the signal support of each measure (manuscript Section 2.4): **A**,
structure speech production cannot generate; **B**, measured where target speech is absent;
**C**, measured on, or relative to, the speech signal.

In [ ]:
SEPARABILITY_CLASS = {
    'qadd_pause_ac_level_dbfs_median': 'B',
    'qadd_pause_level_iqr_db': 'B',
    'qadd_speech_pause_level_contrast_db': 'C',
    'qadd_pause_spectral_flatness': 'B',
    'qadd_mains_hum_comb_score_db': 'A',
    'qgain_typical_speech_level_dbfs': 'C',
    'qgain_within_segment_iqr_db': 'C',
    'qgain_between_segment_mad_db': 'C',
    'qgain_abs_drift_db_per_min': 'C',
    'qrev_tail_excess_100ms_db': 'C',
    'qrev_tail_persistence_median_sec': 'C',
    'qrev_downward_decay_rate_db_per_sec': 'C',
    'qrev_srmr_norm': 'C',
    'qchan_ltas_distance_db': 'C',
    'qchan_rolloff95_deficit_hz': 'C',
    'qchan_highband_ratio_deficit': 'C',
    'qchan_tilt_steepening_db_per_oct': 'C',
    'qdist_hard_clipped_sample_fraction': 'A',
    'qdist_hard_clip_event_rate_per_min': 'A',
    'qdist_hard_clipped_frame_fraction': 'A',
}
registry = FEATURE_REGISTRY.copy()
registry['separability_class'] = registry['feature'].map(SEPARABILITY_CLASS)

MAIN_OUTCOME_CLASSES = {'continuous', 'censored_continuous', 'one_sided_zero'}
main_registry = registry.loc[
    registry['include_main'] & registry['outcome_class'].isin(MAIN_OUTCOME_CLASSES)
].copy()
MAIN_FEATURES = main_registry['feature'].tolist()
if len(MAIN_FEATURES) != 14:
    raise AssertionError(f'Expected 14 main outputs, found {len(MAIN_FEATURES)}')

# Core vs supporting is the inference tier frozen in Notebook 02's registry; it follows the
# validated role assigned during analytical validation (primary -> core).
role_table = registry[['feature', 'family', 'label', 'validated_role', 'inference_tier',
                       'include_main', 'separability_class']]
export(role_table, 'R0_feature_roles_and_classes.csv')
display(main_registry[['family', 'label', 'inference_tier', 'separability_class']])
print(main_registry['separability_class'].value_counts().to_dict())

## R1. Normal-bulbar ALS versus controls

Same model routes and scaling as the Notebook 02 cohort analysis, restricted to (primary) ALS recordings
with a recording-matched ALSFRS-R bulbar score of 11–12, or (sensitivity) ALS participants whose mean
matched bulbar score is ≥ 11. Control recordings are unchanged. Benjamini–Hochberg correction is applied
within analysis × inference tier, as in Notebook 02.

In [ ]:
def cohort_models_for(subset: pd.DataFrame, label: str) -> pd.DataFrame:
    rows = []
    for row in main_registry.itertuples(index=False):
        feature = row.feature
        d = subset.loc[subset[f'{feature}__available']].copy()
        if row.outcome_class in ['continuous', 'censored_continuous']:
            result, _ = fit_continuous_clustered(
                d, f'{feature}__z', 'cohort_als', feature=feature,
                analysis_name='cohort_continuous', target_terms=['cohort_als'],
            )
        else:
            d['positive_presence'] = d[f'{feature}__positive'].astype(int)
            result, _ = fit_binary_gee(
                d, 'positive_presence', 'cohort_als', feature=feature,
                analysis_name='cohort_presence', target_terms=['cohort_als'],
                enforce_cohort_support=True,
            )
        for item in result:
            item['n_als_participants'] = d.loc[d['cohort'] == 'ALS', 'participant_id'].nunique()
            item['n_control_participants'] = d.loc[d['cohort'] == 'CONTROL', 'participant_id'].nunique()
        rows.extend(result)
    frame = pd.DataFrame(rows).merge(
        main_registry[['feature', 'family', 'label', 'inference_tier', 'separability_class']],
        on='feature', how='left',
    )
    frame['q_value'] = frame.groupby(['analysis', 'inference_tier'])['p_value'].transform(bh)
    frame['subset'] = label
    return frame


controls = analysis['cohort'].eq('CONTROL')
als = analysis['cohort'].eq('ALS')
recording_normal = als & analysis['bulbar_score_matched'].ge(NORMAL_BULBAR_MIN)
participant_mean_bulbar = analysis.loc[als].groupby('participant_id')['bulbar_score_matched'].mean()
normal_participants = participant_mean_bulbar.index[participant_mean_bulbar.ge(NORMAL_BULBAR_MIN)]
participant_normal = als & analysis['participant_id'].isin(normal_participants)

subsets = {
    'normal_bulbar_recordings': analysis.loc[controls | recording_normal],
    'normal_bulbar_participants': analysis.loc[controls | participant_normal],
}
r1_population = pd.DataFrame([
    {
        'subset': name,
        'als_recordings': int(frame['cohort'].eq('ALS').sum()),
        'als_participants': int(frame.loc[frame['cohort'] == 'ALS', 'participant_id'].nunique()),
        'control_recordings': int(frame['cohort'].eq('CONTROL').sum()),
        'control_participants': int(frame.loc[frame['cohort'] == 'CONTROL', 'participant_id'].nunique()),
    }
    for name, frame in subsets.items()
])
display(r1_population)

with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    r1_models = pd.concat([cohort_models_for(frame, name) for name, frame in subsets.items()],
                          ignore_index=True)

full_cohort = pd.read_csv(NB02_TABLES / 'cohort_models.csv')
full_cohort = full_cohort.loc[
    full_cohort['analysis'].isin(['cohort_continuous', 'cohort_presence'])
    & full_cohort['feature'].isin(MAIN_FEATURES),
    ['feature', 'analysis', 'estimate', 'ci95_low', 'ci95_high', 'q_value'],
].rename(columns={'estimate': 'full_estimate', 'ci95_low': 'full_ci95_low',
                  'ci95_high': 'full_ci95_high', 'q_value': 'full_q_value'})

r1_comparison = r1_models.merge(full_cohort, on=['feature', 'analysis'], how='left')
r1_comparison['full_significant'] = r1_comparison['full_q_value'].lt(Q_THRESHOLD)
r1_comparison['subset_significant'] = r1_comparison['q_value'].lt(Q_THRESHOLD)
r1_comparison['same_direction'] = np.sign(r1_comparison['estimate']).eq(np.sign(r1_comparison['full_estimate']))
r1_comparison['estimate_ratio_subset_to_full'] = r1_comparison['estimate'] / r1_comparison['full_estimate']
r1_comparison['scale'] = np.where(r1_comparison['analysis'].eq('cohort_presence'),
                                  'log-odds (ALS vs control)', 'IQR-standardized difference')
r1_columns = ['subset', 'family', 'label', 'separability_class', 'inference_tier', 'scale',
              'full_estimate', 'full_ci95_low', 'full_ci95_high', 'full_q_value', 'full_significant',
              'estimate', 'ci95_low', 'ci95_high', 'q_value', 'subset_significant',
              'same_direction', 'estimate_ratio_subset_to_full', 'n_recordings',
              'n_als_participants', 'n_control_participants', 'model']
r1_comparison = r1_comparison[r1_columns]
export(r1_population, 'R1_population.csv')
export(r1_comparison, 'R1_normal_bulbar_vs_controls.csv')
display(r1_comparison.loc[r1_comparison['subset'] == 'normal_bulbar_recordings'])

## R2. Availability by cohort and bulbar severity group

Descriptive availability for every support-dependent measure (available in < 99% of recordings), by
cohort and recording-matched bulbar group. The clustered availability models from Notebook 02
(cohort; between- and within-participant bulbar) are carried over, with odds ratios added.

In [ ]:
def bulbar_group(row: pd.Series) -> str:
    if row['cohort'] == 'CONTROL':
        return 'Control'
    score = row['bulbar_score_matched']
    if pd.isna(score):
        return 'ALS, bulbar missing'
    if score >= 11:
        return 'ALS normal (11-12)'
    if score >= 9:
        return 'ALS mild (9-10)'
    if score >= 6:
        return 'ALS moderate (6-8)'
    return 'ALS severe (<=5)'


analysis = analysis.copy()  # de-fragment before adding columns
GROUP_ORDER = ['Control', 'ALS normal (11-12)', 'ALS mild (9-10)', 'ALS moderate (6-8)',
               'ALS severe (<=5)', 'ALS, bulbar missing']
analysis['bulbar_group'] = pd.Categorical(analysis.apply(bulbar_group, axis=1),
                                          categories=GROUP_ORDER, ordered=True)

support_dependent = [
    feature for feature in registry['feature']
    if 0.01 < analysis[f'{feature}__available'].mean() < 0.99
]
rows = []
for feature in support_dependent:
    meta = registry.set_index('feature').loc[feature]
    for group, frame in analysis.groupby('bulbar_group', observed=False):
        n = len(frame)
        k = int(frame[f'{feature}__available'].sum())
        rows.append({'family': meta['family'], 'label': meta['label'],
                     'separability_class': meta['separability_class'], 'group': group,
                     'n_recordings': n, 'n_available': k,
                     'percent_available': 100 * k / n if n else np.nan})
r2_descriptive = pd.DataFrame(rows)
r2_wide = r2_descriptive.pivot_table(
    index=['family', 'label', 'separability_class'], columns='group',
    values='percent_available', observed=False,
).reindex(columns=GROUP_ORDER).round(1).reset_index()
group_n = analysis['bulbar_group'].value_counts().reindex(GROUP_ORDER)

availability_models = pd.read_csv(NB02_TABLES / 'availability_models.csv')
availability_models = availability_models.merge(
    registry[['feature', 'label', 'separability_class']], on='feature', how='left')
availability_models['odds_ratio'] = np.exp(availability_models['estimate'])
availability_models['or_ci95_low'] = np.exp(availability_models['ci95_low'])
availability_models['or_ci95_high'] = np.exp(availability_models['ci95_high'])
availability_models['interpretation'] = np.select(
    [availability_models['target_term'].eq('cohort_als'),
     availability_models['target_term'].eq('bulbar_between'),
     availability_models['target_term'].eq('bulbar_within')],
    ['OR of availability, ALS vs control',
     'OR per 1-point higher participant-mean bulbar score (better function)',
     'OR per 1-point higher recording-specific bulbar score (better function)'],
    default='',
)

export(r2_descriptive, 'R2_availability_by_group_long.csv')
export(r2_wide, 'R2_availability_by_group.csv')
export(group_n.rename_axis('group').reset_index(name='n_recordings'), 'R2_group_sizes.csv')
export(availability_models, 'R2_availability_models.csv')
display(group_n)
display(r2_wide)
display(availability_models[['label', 'separability_class', 'analysis', 'target_term',
                             'odds_ratio', 'or_ci95_low', 'or_ci95_high', 'q_value']])

## R3. Effects and persistence by separability class, against the base rate

Uses Notebook 02's primary cohort and bulbar models for the 14 main outputs (q < 0.05 within
Notebook 02's prespecified FDR families). The hypergeometric probability asks how often at least the
observed number of significant effects would fall in class C if the same number of significant
effects were placed at random among the tests.

In [ ]:
cohort_models = pd.read_csv(NB02_TABLES / 'cohort_models.csv')
bulbar_models = pd.read_csv(NB02_TABLES / 'bulbar_models.csv')

cohort_tests = cohort_models.loc[
    cohort_models['analysis'].isin(['cohort_continuous', 'cohort_presence'])
    & cohort_models['feature'].isin(MAIN_FEATURES)
].assign(test='cohort')
bulbar_tests = bulbar_models.loc[
    bulbar_models['analysis'].isin(['bulbar_continuous', 'bulbar_presence'])
    & bulbar_models['feature'].isin(MAIN_FEATURES)
].copy()
bulbar_tests['test'] = bulbar_tests['target_term'].map(
    {'bulbar_between': 'bulbar_between', 'bulbar_within': 'bulbar_within'})

tests = pd.concat([cohort_tests, bulbar_tests], ignore_index=True)
tests = tests.merge(main_registry[['feature', 'label', 'separability_class']], on='feature', how='left')
tests['significant'] = tests['q_value'].lt(Q_THRESHOLD)

observed_counts = tests.groupby('test')['significant'].sum().to_dict()
MANUSCRIPT_COUNTS = {'cohort': 6, 'bulbar_between': 4, 'bulbar_within': 2}
count_check = pd.DataFrame([
    {'test': test, 'manuscript_v16': expected, 'this_run': int(observed_counts.get(test, 0)),
     'matches': int(observed_counts.get(test, 0)) == expected}
    for test, expected in MANUSCRIPT_COUNTS.items()
])
display(count_check)
if not count_check['matches'].all():
    warnings.warn('Significant-effect counts differ from manuscript v16; report this run\'s values.',
                  RuntimeWarning)

by_class = (
    tests.groupby('separability_class')
    .agg(n_tests=('significant', 'size'), n_significant=('significant', 'sum'))
    .reindex(['A', 'B', 'C']).fillna(0).astype(int).reset_index()
)
by_class['percent_significant'] = 100 * by_class['n_significant'] / by_class['n_tests'].replace(0, np.nan)
by_class_test = pd.crosstab([tests['test'], tests['separability_class']], tests['significant'])
by_class_test = by_class_test.reindex(columns=[False, True], fill_value=0).reset_index()
by_class_test.columns = ['test', 'separability_class', 'n_not_significant', 'n_significant']

N_total = len(tests)
K_classC = int(tests['separability_class'].eq('C').sum())
n_sig = int(tests['significant'].sum())
k_sig_C = int((tests['significant'] & tests['separability_class'].eq('C')).sum())
p_hyper = float(stats.hypergeom.sf(k_sig_C - 1, N_total, K_classC, n_sig)) if n_sig else np.nan
fisher_table = [[k_sig_C, K_classC - k_sig_C],
                [n_sig - k_sig_C, (N_total - K_classC) - (n_sig - k_sig_C)]]
fisher_or, fisher_p = stats.fisher_exact(fisher_table, alternative='greater')
base_rate = pd.DataFrame([{
    'n_tests': N_total, 'n_tests_class_C': K_classC, 'n_significant': n_sig,
    'n_significant_class_C': k_sig_C,
    'expected_class_C_if_random': n_sig * K_classC / N_total if N_total else np.nan,
    'hypergeometric_p_at_least_observed': p_hyper,
    'fisher_odds_ratio_C_vs_AB': fisher_or, 'fisher_p_one_sided': fisher_p,
}])

persistence = pd.read_csv(NB02_TABLES / 'participant_level_persistence.csv')
persistence = persistence.merge(registry[['feature', 'separability_class', 'include_main']],
                                on='feature', how='left')
persistence_table = persistence[['family', 'label', 'separability_class', 'include_main', 'icc',
                                 'icc_ci95_low', 'icc_ci95_high', 'n_recordings',
                                 'n_participants', 'n_repeated_participants']].sort_values(
    ['separability_class', 'icc'], ascending=[True, False])
persistence_by_class = (
    persistence.groupby('separability_class')['icc']
    .agg(n_measures='count', median_icc='median', min_icc='min', max_icc='max').reset_index()
)

export(tests[['test', 'feature', 'label', 'separability_class', 'analysis', 'estimate',
              'ci95_low', 'ci95_high', 'p_value', 'q_value', 'significant']], 'R3_tests_by_class.csv')
export(count_check, 'R3_count_check_vs_manuscript.csv')
export(by_class, 'R3_effects_by_class.csv')
export(by_class_test, 'R3_effects_by_class_and_test.csv')
export(base_rate, 'R3_base_rate_test.csv')
export(persistence_table, 'R3_persistence_icc_by_measure.csv')
export(persistence_by_class, 'R3_persistence_by_class.csv')
display(by_class)
display(base_rate)
display(persistence_by_class)

## R4. Family-organization contrast: shared-support and leave-one-family-out sensitivities

Recomputed from Notebook 03's pairwise table using its estimator (mean |ρ| within family minus mean
|ρ| between families) and label permutation preserving family sizes. Variants:

* **original**: Notebook 03's structural-dependency exclusions (must reproduce Notebook 03's observed contrast);
* **shared_support**: additionally excludes pairs computed from the same frames or the same reference spectrum
  (pause-region level × pause-level variability; all pairs among the three one-sided QCHAN deficits);
* **leave out <family>**: original exclusions, with every pair involving that family removed.

In [ ]:
pairs = pd.read_csv(NB03_TABLES / 'pairwise_relationship_classification.csv')
nb03_summary = pd.read_csv(NB03_TABLES / 'family_contrast_summary.csv')
family_of = dict(zip(pairs['feature_x'], pairs['family_x'])) | dict(zip(pairs['feature_y'], pairs['family_y']))
features_all = sorted(family_of)

SHARED_SUPPORT_EXTRA = {
    frozenset({'qadd_pause_ac_level_dbfs_median', 'qadd_pause_level_iqr_db'}),
    frozenset({'qchan_rolloff95_deficit_hz', 'qchan_highband_ratio_deficit'}),
    frozenset({'qchan_rolloff95_deficit_hz', 'qchan_tilt_steepening_db_per_oct'}),
    frozenset({'qchan_highband_ratio_deficit', 'qchan_tilt_steepening_db_per_oct'}),
}
pair_key = pairs.apply(lambda r: frozenset({r['feature_x'], r['feature_y']}), axis=1)
structural = pairs['structural_dependency'].astype(str).str.lower().isin(['true', '1'])


def contrast_and_permutation(mask: pd.Series, value_col: str, seed: int) -> dict:
    sub = pairs.loc[mask & np.isfinite(pairs[value_col])]
    feats = sorted(set(sub['feature_x']) | set(sub['feature_y']))
    labels = np.asarray([family_of[f] for f in feats])
    values = sub[value_col].abs().to_numpy()
    fx, fy = sub['feature_x'].to_numpy(), sub['feature_y'].to_numpy()

    def delta(mapping: dict) -> float:
        same = np.fromiter((mapping[a] == mapping[b] for a, b in zip(fx, fy)), bool, len(fx))
        if not same.any() or same.all():
            return np.nan
        return float(values[same].mean() - values[~same].mean())

    observed = delta(dict(zip(feats, labels)))
    generator = np.random.default_rng(seed)
    null = np.array([delta(dict(zip(feats, generator.permutation(labels))))
                     for _ in range(N_FAMILY_PERMUTATIONS)])
    null = null[np.isfinite(null)]
    same_obs = np.fromiter((family_of[a] == family_of[b] for a, b in zip(fx, fy)), bool, len(fx))
    return {
        'n_features': len(feats), 'n_within_pairs': int(same_obs.sum()),
        'n_between_pairs': int((~same_obs).sum()), 'delta': observed,
        'permutation_p': (1 + np.sum(null >= observed)) / (1 + len(null)) if len(null) else np.nan,
        'n_permutations': len(null),
    }


variants = {'original': ~structural,
            'shared_support': ~structural & ~pair_key.isin(SHARED_SUPPORT_EXTRA)}
for family in sorted(set(family_of.values())):
    variants[f'leave out {family}'] = ~structural & pairs['family_x'].ne(family) & pairs['family_y'].ne(family)

rows = []
for v_index, (variant, mask) in enumerate(variants.items()):
    for level in ['between', 'within']:
        result = contrast_and_permutation(mask, f'{level}_rho', SEED + 100 * v_index + (level == 'within'))
        rows.append({'variant': variant, 'level': level, **result})
r4 = pd.DataFrame(rows)

for level in ['between', 'within']:
    reproduced = r4.loc[(r4['variant'] == 'original') & (r4['level'] == level), 'delta'].iloc[0]
    reference = nb03_summary.loc[nb03_summary['level'] == level, 'delta'].iloc[0]
    if not np.isclose(reproduced, reference, atol=1e-9):
        raise AssertionError(f'{level}: reproduced delta {reproduced} != Notebook 03 delta {reference}')
print('Original contrast reproduces Notebook 03 at both levels.')
export(r4, 'R4_family_contrast_sensitivity.csv')
display(r4)

## 5. Summary for the manuscript

In [ ]:
def fmt(x, digits=2):
    return '—' if pd.isna(x) else f'{x:.{digits}f}'


primary_r1 = r1_comparison.loc[r1_comparison['subset'] == 'normal_bulbar_recordings']
pop = r1_population.set_index('subset').loc['normal_bulbar_recordings']
lines = [
    '# Revision analyses: summary',
    f'Generated {datetime.now(timezone.utc).isoformat(timespec="seconds")} · FINAL_RUN={FINAL_RUN} · '
    f'Python {platform.python_version()} · pandas {pd.__version__}',
    '',
    '## R1. Normal-bulbar ALS (11-12) vs controls',
    f'ALS {pop.als_recordings} recordings / {pop.als_participants} participants; '
    f'controls {pop.control_recordings} / {pop.control_participants}.',
    '',
    '| Measure | Class | Full cohort est. (q) | Normal-bulbar est. [95% CI] (q) | Same direction |',
    '|---|---|---|---|---|',
]
for r in primary_r1.itertuples(index=False):
    lines.append(
        f'| {r.label} | {r.separability_class} | {fmt(r.full_estimate)} ({fmt(r.full_q_value, 3)}) | '
        f'{fmt(r.estimate)} [{fmt(r.ci95_low)}, {fmt(r.ci95_high)}] ({fmt(r.q_value, 3)}) | {r.same_direction} |'
    )
lines += ['', f'Full-cohort significant: {int(primary_r1.full_significant.sum())}; '
          f'normal-bulbar significant: {int(primary_r1.subset_significant.sum())}; '
          f'of full-cohort effects, still significant: '
          f'{int((primary_r1.full_significant & primary_r1.subset_significant).sum())}.', '']

lines += ['## R2. Availability (%) by group', '',
          '| Measure | Class | ' + ' | '.join(f'{g} (n={group_n[g]})' for g in GROUP_ORDER) + ' |',
          '|---|---|' + '---|' * len(GROUP_ORDER)]
for r in r2_wide.itertuples(index=False):
    values = list(r)[3:]
    lines.append(f'| {r.label} | {r.separability_class} | ' + ' | '.join(fmt(v, 1) for v in values) + ' |')
sig_avail = availability_models.loc[availability_models['q_value'].lt(Q_THRESHOLD)]
lines += ['', 'Availability models with q < 0.05:' if len(sig_avail) else 'No availability model had q < 0.05.']
for r in sig_avail.itertuples(index=False):
    lines.append(f'- {r.label}: {r.interpretation} = {fmt(r.odds_ratio)} '
                 f'[{fmt(r.or_ci95_low)}, {fmt(r.or_ci95_high)}], q = {fmt(r.q_value, 3)}')

b = base_rate.iloc[0]
lines += ['', '## R3. Effects by separability class', '',
          '| Class | Tests | Significant | % |', '|---|---|---|---|']
for r in by_class.itertuples(index=False):
    lines.append(f'| {r.separability_class} | {r.n_tests} | {r.n_significant} | {fmt(r.percent_significant, 0)} |')
lines += ['', f'{int(b.n_significant_class_C)} of {int(b.n_significant)} significant effects were class C; '
          f'{fmt(b.expected_class_C_if_random, 1)} expected at random '
          f'(class C = {int(b.n_tests_class_C)}/{int(b.n_tests)} tests). '
          f'Hypergeometric P(≥ observed) = {fmt(b.hypergeometric_p_at_least_observed, 3)}; '
          f'one-sided Fisher OR = {fmt(b.fisher_odds_ratio_C_vs_AB)}, p = {fmt(b.fisher_p_one_sided, 3)}.',
          '', 'Count check vs manuscript v16: '
          + ', '.join(f'{r.test} {r.this_run} (v16 {r.manuscript_v16})' for r in count_check.itertuples()),
          '', '### Persistence (ICC) by class', '', '| Class | Measures | Median ICC | Range |', '|---|---|---|---|']
for r in persistence_by_class.itertuples(index=False):
    lines.append(f'| {r.separability_class} | {r.n_measures} | {fmt(r.median_icc)} | {fmt(r.min_icc)}–{fmt(r.max_icc)} |')
lines += ['', '| Measure | Class | ICC [95% CI] |', '|---|---|---|']
for r in persistence_table.itertuples(index=False):
    lines.append(f'| {r.label} | {r.separability_class} | {fmt(r.icc)} [{fmt(r.icc_ci95_low)}, {fmt(r.icc_ci95_high)}] |')

lines += ['', '## R4. Family contrast sensitivity', '',
          '| Variant | Level | Within pairs | Between pairs | Δ | Permutation p |', '|---|---|---|---|---|---|']
for r in r4.itertuples(index=False):
    lines.append(f'| {r.variant} | {r.level} | {r.n_within_pairs} | {r.n_between_pairs} | {fmt(r.delta, 3)} | {fmt(r.permutation_p, 4)} |')

summary_path = OUTPUT_ROOT / 'revision_summary.md'
summary_path.write_text('\n'.join(lines) + '\n', encoding='utf-8')

manifest = {
    'generated_utc': datetime.now(timezone.utc).isoformat(timespec='seconds'),
    'final_run': FINAL_RUN, 'seed': SEED, 'n_family_permutations': N_FAMILY_PERMUTATIONS,
    'normal_bulbar_min': NORMAL_BULBAR_MIN, 'q_threshold': Q_THRESHOLD,
    'inputs': {name: str(path) for name, path in REQUIRED_INPUTS.items()},
    'outputs': sorted(str(p.relative_to(OUTPUT_ROOT)) for p in TABLE_DIR.glob('*.csv')),
}
(LOG_DIR / 'run_manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
print(summary_path.read_text(encoding='utf-8'))